# 1주차 금융 선택과제 · 학생용 빈칸

## 과제의 업무 목표
가맹점 기준표를 정제하고 운영 규칙표를 붙여, 수동 검토 필요 여부를 예측할 입력표를 준비합니다.

결혼정보회사 수업에서 `member_id`를 기준으로 회원표를 다뤘다면, 이 과제에서는 `merchant_id`를 기준으로 같은 절차를 적용합니다.

진행 순서: **파일 확인 → 중복·문자·숫자 정리 → 다른 표 결합 → 날짜 계산 → X와 y 분리 → train/test 분할 → 전처리 → 새 데이터 적용**

학생용 노트북은 `FILL_ME`만 채웁니다. 실행 전에 예상 결과를 먼저 적고, 실행 뒤에는 `assert`가 통과하는지 확인하세요.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

def find_assignment_dir():
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        candidates = [
            base,
            base / "course_materials" / "assignments" / "week1" / "option2_finance",
            base / "assignments" / "week1" / "option2_finance",
        ]
        for candidate in candidates:
            if (candidate / "data" / "finance_merchants.csv").exists():
                return candidate
    raise FileNotFoundError("finance_merchants.csv를 찾지 못했습니다. assignments 폴더가 포함된 프로젝트를 VS Code에서 열어 주세요.")

ASSIGNMENT_DIR = find_assignment_dir()
DATA_DIR = ASSIGNMENT_DIR / "data"
OUTPUT_DIR = ASSIGNMENT_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
print("과제 폴더:", ASSIGNMENT_DIR)

## Q1. 두 표를 읽기 전에 파일 이름과 역할부터 확인합니다

- 기준표의 한 행은 대상 하나입니다.
- 보조표는 같은 ID에 연결할 기준이나 규칙을 담고 있습니다.
- 예상: 원본 기준표 243행, 보조표 240행입니다.

In [ ]:
main_path = DATA_DIR / "finance_merchants.csv"
extra_path = DATA_DIR / "finance_risk_rules.csv"
print(main_path.name, main_path.exists())
print(extra_path.name, extra_path.exists())

raw = pd.FILL_ME(main_path)
extra = pd.FILL_ME(extra_path)
print(raw.shape, extra.shape)
assert raw.shape[0] == 243
assert extra.shape[0] == 240

## Q2. 중복 ID와 계산할 수 없는 값을 정리합니다

원본에는 같은 `merchant_id`가 3번 더 들어 있고, `monthly_sales_10k_krw`에는 숫자가 아닌 글자가 섞여 있습니다. 원본 `raw`는 남기고 `clean` 복사본을 정리합니다.

In [ ]:
clean = raw.FILL_ME("merchant_id", keep="first").copy()
clean["region"] = clean["region"].str.strip().replace({"Seoul": "서울", "seoul": "서울"})
clean["monthly_sales_10k_krw"] = pd.FILL_ME(clean["monthly_sales_10k_krw"], errors="FILL_ME")
clean["monthly_sales_10k_krw"] = clean["monthly_sales_10k_krw"].fillna(clean["monthly_sales_10k_krw"].median())
clean["refund_rate"] = clean["refund_rate"].fillna(clean["refund_rate"].median())

print("행 수:", len(clean))
print("중복 ID:", clean["merchant_id"].duplicated().sum())
print("계산용 열의 결측:", clean[["monthly_sales_10k_krw", "refund_rate"]].isna().sum().sum())
assert len(clean) == 240
assert clean["merchant_id"].is_unique

## Q3. ID가 같은 행끼리 보조표를 붙입니다

행 순서나 이름이 아니라 `merchant_id`가 같은 대상을 연결합니다. `validate="one_to_one"`은 양쪽 표에서 ID가 한 번씩만 나타나는지 확인합니다.

In [ ]:
joined = clean.FILL_ME(extra, on="merchant_id", how="FILL_ME", validate="FILL_ME")
print(joined.shape)
print(joined[['daily_limit_10k_krw', 'cross_border_allowed', 'refund_alert_rate', 'chargeback_alert_rate', 'review_cycle_days']].head(2))
assert len(joined) == 240
assert joined[['daily_limit_10k_krw', 'cross_border_allowed', 'refund_alert_rate', 'chargeback_alert_rate', 'review_cycle_days']].isna().sum().sum() == 0

## Q4. 날짜 글자를 날짜 자료형으로 바꾸고 경과일을 만듭니다

날짜를 빼려면 먼저 `datetime`으로 바꿔야 합니다. 바꾸지 못한 날짜는 `NaT`가 됩니다. 기준일은 결과를 다시 실행해도 바뀌지 않도록 `2026-09-01`로 고정합니다.

In [ ]:
joined["event_dt"] = pd.FILL_ME(joined["last_kyc_review_date"], errors="FILL_ME")
reference_date = pd.Timestamp("2026-09-01")
joined["days_since_kyc_review"] = (reference_date - joined["event_dt"]).dt.days
joined["days_since_kyc_review"] = joined["days_since_kyc_review"].fillna(joined["days_since_kyc_review"].median())
print(joined[["merchant_id", "last_kyc_review_date", "days_since_kyc_review"]].head(3))
assert joined["days_since_kyc_review"].isna().sum() == 0

## Q5. 모델이 볼 X와 확인할 정답 y를 나누고 train/test로 분할합니다

`X`에는 입력 정보 12개 열을, `y`에는 `manual_review_required` 한 열을 둡니다. 같은 대상의 X와 y가 함께 움직이도록 `train_test_split`에 동시에 넣습니다.

In [ ]:
from sklearn.model_selection import train_test_split

numeric_features = ['monthly_sales_10k_krw', 'avg_ticket_krw', 'refund_rate', 'chargeback_rate', 'support_tickets_90d', 'days_since_kyc_review', 'daily_limit_10k_krw', 'review_cycle_days']
categorical_features = ['industry', 'region', 'kyc_status', 'risk_tier']
input_features = numeric_features + categorical_features

X = joined[input_features].copy()
y = joined["manual_review_required"].copy()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=FILL_ME, random_state=FILL_ME, stratify=FILL_ME
)
print(X.shape, y.shape)
print(X_train.shape, X_test.shape)
assert X_train.shape == (192, 12)
assert X_test.shape == (48, 12)

## Q6. 숫자와 문자에 다른 규칙을 적용하는 전처리기를 만듭니다

- 숫자: 결측을 중앙값으로 채운 뒤 크기를 맞춥니다.
- 문자: 가장 자주 나온 값으로 채운 뒤 0과 1 열로 바꿉니다.
- `fit_transform`은 train에서 기준을 배우며 적용하고, `transform`은 배운 기준만 test에 적용합니다.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="FILL_ME")),
    ("scaler", StandardScaler()),
])
categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="FILL_ME")),
    ("onehot", OneHotEncoder(handle_unknown="FILL_ME", sparse_output=False)),
])
preprocessor = ColumnTransformer([
    ("num", numeric_pipe, numeric_features),
    ("cat", categorical_pipe, categorical_features),
])

train_processed = preprocessor.FILL_ME(X_train)
test_processed = preprocessor.FILL_ME(X_test)
print(train_processed.shape, test_processed.shape)
assert train_processed.shape[0] == 192
assert test_processed.shape[0] == 48
assert train_processed.shape[1] == test_processed.shape[1]
assert np.isnan(train_processed).sum() == 0

## Q7. 처음 보는 40행에도 같은 기준을 적용하고 결과를 저장합니다

새 데이터에서는 다시 `fit`하지 않습니다. 원본의 열을 정리한 뒤, train에서 이미 기준을 배운 `preprocessor`의 `transform`만 사용합니다.

In [ ]:
new_raw = pd.read_csv(DATA_DIR / "finance_merchants_new.csv")
new_clean = new_raw.copy()
new_clean["region"] = new_clean["region"].str.strip().replace({"Seoul": "서울", "seoul": "서울"})
new_clean["monthly_sales_10k_krw"] = pd.to_numeric(new_clean["monthly_sales_10k_krw"], errors="coerce")
new_clean["refund_rate"] = pd.to_numeric(new_clean["refund_rate"], errors="coerce")
new_clean["event_dt"] = pd.to_datetime(new_clean["last_kyc_review_date"], errors="coerce")
new_clean["days_since_kyc_review"] = (pd.Timestamp("2026-09-01") - new_clean["event_dt"]).dt.days

# 보조표에만 있던 숫자 열은 기존 보조표의 중앙값을 사용합니다.
for column in ['daily_limit_10k_krw', 'cross_border_allowed', 'refund_alert_rate', 'chargeback_alert_rate', 'review_cycle_days']:
    if column not in new_clean.columns:
        new_clean[column] = joined[column].median() if pd.api.types.is_numeric_dtype(joined[column]) else joined[column].mode()[0]

new_X = new_clean[input_features].copy()
new_processed = preprocessor.FILL_ME(new_X)

output_path = OUTPUT_DIR / "finance_merchant_master_clean.csv"
joined.to_csv(output_path, index=False, encoding="utf-8-sig")
print(new_X.shape, new_processed.shape)
print("저장:", output_path)
assert new_X.shape == (40, 12)
assert new_processed.shape[1] == train_processed.shape[1]
assert output_path.exists()

## 제출 전 확인

1. 모든 `FILL_ME`를 지웠습니다.
2. 모든 `assert`가 조용히 통과했습니다.
3. 실행 결과가 포함된 자신의 노트북을 저장했습니다.
4. `outputs` 폴더의 정제 CSV를 함께 제출합니다.
5. 가장 헷갈렸던 단계와 해결 방법을 Markdown 셀에 3문장으로 적습니다.